# 🐍 Buổi 2 — Python for Everyone
**Big Data Engineering K16**

---

| Bài | Chủ đề | Thời gian |
|-----|--------|-----------|
| 1 | Khám phá Python & Ecosystem | 10 phút |
| 2 | 3 phong cách xử lý dữ liệu | 15 phút |
| 3 | Gọi API & xử lý JSON | 20 phút |
| 4 | Hệ sinh thái Data Engineering | 10 phút |

> **Cách chạy:** Nhấn `Shift+Enter` để chạy từng cell.

## 0. Khởi động — Chạy cell này trước

In [ ]:
import sys
print(f"Python version: {sys.version}")
print("✅ Môi trường sẵn sàng!")

---
## Bài 1 — Khám Phá Python & Ecosystem (10 phút)

Trước khi viết code, hãy hiểu mình đang làm việc với công cụ gì.

### 1.1 — Triết lý Python

Chạy cell dưới và **đọc kỹ** 19 nguyên tắc. Sau đó trả lời câu hỏi.

In [ ]:
import this

**Câu hỏi:** Theo bạn, nguyên tắc nào quan trọng nhất với một Data Engineer? Tại sao?

> ✏️ **Trả lời của bạn (double-click để chỉnh sửa):**
>
> ...

### 1.2 — Kiểm tra môi trường

In [ ]:
import sys
import platform
import pkg_resources

print(f"Python version : {sys.version.split()[0]}")
print(f"Platform       : {platform.system()} {platform.release()}")
print(f"Executable     : {sys.executable}")

print("\n--- Thư viện Data Engineering đã cài ---")
de_libs = ["pandas", "numpy", "matplotlib", "requests"]
for lib in de_libs:
    try:
        version = pkg_resources.get_distribution(lib).version
        print(f"  ✓ {lib:<12} {version}")
    except Exception:
        print(f"  ✗ {lib:<12} chưa cài")

---
## Bài 2 — Ba Phong Cách Xử Lý Dữ Liệu (15 phút)

Cùng một tác vụ — **tính tổng doanh thu từ file JSON** — được giải theo 3 cách khác nhau.

Mục tiêu: hiểu tại sao **Pythonic** và **pandas** được ưa dùng trong Data Engineering.

### 2.0 — Tạo dữ liệu mẫu

In [ ]:
import json

sales_data = [
    {"product": "Laptop",   "price": 1200, "qty": 3},
    {"product": "Mouse",    "price": 25,   "qty": 15},
    {"product": "Keyboard", "price": 75,   "qty": 8},
    {"product": "Monitor",  "price": 350,  "qty": 5},
    {"product": "Headset",  "price": 90,   "qty": 12},
]

with open("/workspace/data/sales.json", "w") as f:
    json.dump(sales_data, f, indent=2)

print("✅ Đã tạo /workspace/data/sales.json")
print(f"   {len(sales_data)} bản ghi")

### 2.1 — Phong cách Imperative (kiểu Java/C)

In [ ]:
import json

with open("/workspace/data/sales.json") as f:
    data = json.load(f)

total = 0
for item in data:
    total = total + item["price"] * item["qty"]

print(f"Tổng doanh thu (imperative): {total:,} USD")

### 2.2 — Phong cách Pythonic

In [ ]:
import json

with open("/workspace/data/sales.json") as f:
    data = json.load(f)

total = sum(item["price"] * item["qty"] for item in data)
print(f"Tổng doanh thu (pythonic): {total:,} USD")

### 2.3 — Phong cách Data Engineering (pandas)

In [ ]:
import pandas as pd

df = pd.read_json("/workspace/data/sales.json")
df["revenue"] = df["price"] * df["qty"]

print(df.to_string(index=False))
print(f"\nTổng doanh thu (pandas): {df['revenue'].sum():,} USD")

**Câu hỏi:**
1. Phong cách nào dễ đọc nhất với bạn?
2. Nếu dataset có 10 triệu dòng, bạn sẽ dùng phong cách nào? Tại sao?

> ✏️ **Trả lời của bạn:**
>
> 1. ...
> 2. ...

---
## Bài 3 — Gọi API & Xử Lý JSON (20 phút)

Data Engineering thường xuyên cần **gọi API để lấy dữ liệu**. Bài này thực hành với API tỷ giá công khai.

### 3.1 — Gọi API và xem cấu trúc dữ liệu

In [ ]:
import requests

url = "https://api.exchangerate-api.com/v4/latest/USD"

try:
    response = requests.get(url, timeout=5)
    print(f"Status code : {response.status_code}")

    if response.status_code == 200:
        data = response.json()
        print(f"Base currency: {data['base']}")
        print(f"Last updated : {data['date']}")
        print(f"Số lượng tỷ giá: {len(data['rates'])} đồng tiền")

        # In một vài key đầu tiên để xem cấu trúc
        sample = dict(list(data['rates'].items())[:5])
        print(f"\nMẫu dữ liệu: {sample}")
    else:
        print("API trả về lỗi, dùng dữ liệu offline bên dưới")
        data = None

except Exception as e:
    print(f"Không kết nối được API: {e}")
    print("→ Dùng dữ liệu offline bên dưới")
    data = None

### 3.2 — Trích xuất tỷ giá quan trọng

In [ ]:
# Dùng data từ API nếu có, không thì dùng offline
if data is None:
    data = {
        "base": "USD",
        "date": "2025-01-01",
        "rates": {"VND": 25450, "EUR": 0.92, "JPY": 157.3, "SGD": 1.34, "CNY": 7.24, "GBP": 0.79}
    }
    print("[Dùng dữ liệu offline]")

currencies = ["VND", "EUR", "JPY", "SGD", "CNY", "GBP"]

print(f"\nTỷ giá so với 1 USD (nguồn: {data['date']}):")
print("-" * 30)
for currency in currencies:
    rate = data["rates"].get(currency, "N/A")
    if isinstance(rate, (int, float)):
        print(f"  1 USD = {rate:>10,.2f} {currency}")
    else:
        print(f"  {currency}: {rate}")

### 3.3 — Bài tập: Chuyển đổi giá sản phẩm sang VND

Dùng tỷ giá lấy được ở trên để chuyển đổi danh sách sản phẩm từ USD sang VND.

In [ ]:
# Tỷ giá (dùng từ data API hoặc mặc định)
usd_vnd_rate = data["rates"].get("VND", 25450)

products_usd = [
    {"name": "Laptop",  "price_usd": 999},
    {"name": "Phone",   "price_usd": 799},
    {"name": "Tablet",  "price_usd": 499},
    {"name": "Monitor", "price_usd": 349},
]

# TODO: Tính price_vnd cho mỗi sản phẩm
# Gợi ý: price_vnd = price_usd * usd_vnd_rate
for product in products_usd:
    product["price_vnd"] = ???  # TODO

# TODO: In bảng kết quả
print(f"Tỷ giá: 1 USD = {usd_vnd_rate:,.0f} VND\n")
print(f"{'Sản phẩm':<12} {'USD':>10} {'VND':>15}")
print("-" * 40)
for product in products_usd:
    print(f"{product['name']:<12} {product['price_usd']:>10,.0f} {product['price_vnd']:>15,.0f}")

### 3.4 — Lưu kết quả ra file JSON

In [ ]:
import json

output = {
    "exchange_date": data["date"],
    "usd_vnd_rate": usd_vnd_rate,
    "products": products_usd
}

with open("/workspace/data/products_vnd.json", "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print("✅ Đã lưu: /workspace/data/products_vnd.json")

# Đọc lại để kiểm tra
with open("/workspace/data/products_vnd.json", encoding="utf-8") as f:
    saved = json.load(f)

print(f"\nKiểm tra — đọc lại file:")
print(f"  Ngày tỷ giá: {saved['exchange_date']}")
print(f"  Số sản phẩm: {len(saved['products'])}")

---
## Bài 4 — Hệ Sinh Thái Data Engineering (10 phút)

Visualize mức độ phổ biến của các công cụ DE để hiểu Python đứng ở đâu trong bức tranh tổng thể.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

tools = {
    "Tool": ["Python", "SQL", "Spark", "Airflow", "Kafka", "dbt", "Hadoop", "Scala"],
    "Score": [95, 90, 75, 65, 60, 55, 40, 35],
    "Category": [
        "Language", "Language", "Processing", "Orchestration",
        "Streaming", "Transformation", "Processing", "Language"
    ]
}

df = pd.DataFrame(tools)
print(df.to_string(index=False))

color_map = {
    "Language":       "#4CAF50",
    "Processing":     "#2196F3",
    "Orchestration":  "#FF9800",
    "Streaming":      "#E91E63",
    "Transformation": "#9C27B0",
}

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(df["Tool"], df["Score"],
              color=[color_map[c] for c in df["Category"]],
              edgecolor="white", linewidth=0.8)

for bar, score in zip(bars, df["Score"]):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 1.5,
            str(score), ha="center", va="bottom", fontsize=10, fontweight="bold")

legend_handles = [
    mpatches.Patch(color=v, label=k) for k, v in color_map.items()
]
ax.legend(handles=legend_handles, loc="upper right", fontsize=9)

ax.set_title("Mức độ phổ biến của công cụ trong Data Engineering (2025)",
             fontsize=13, fontweight="bold", pad=15)
ax.set_ylabel("Điểm phổ biến")
ax.set_ylim(0, 110)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig("/workspace/data/de_tools_popularity.png", dpi=150)
plt.show()
print("\n✅ Đã lưu: /workspace/data/de_tools_popularity.png")

**Câu hỏi:**
1. Tại sao Python và SQL cùng quan trọng? Chúng có thay thế được nhau không?
2. Sau khi học xong Module 4, bạn sẽ làm việc được với bao nhiêu công cụ trong biểu đồ này?

> ✏️ **Trả lời của bạn:**
>
> 1. ...
> 2. ...

---
## Tổng Kết

| Bài | Kỹ năng thực hành |
|-----|-------------------|
| 1 | Import thư viện, kiểm tra môi trường, đọc tài liệu |
| 2 | Đọc JSON, vòng lặp, generator expression, pandas DataFrame |
| 3 | HTTP request, parse JSON, xử lý lỗi, ghi file |
| 4 | pandas, matplotlib, visualize dữ liệu |

**Buổi tiếp theo (3-7):** Đi sâu vào Python — kiểu dữ liệu, hàm, OOP, và Data Engineering patterns.

---
## Nộp Bài

Yêu cầu:
- File notebook `.ipynb` có đầy đủ output của 4 bài
- Các ô **Trả lời của bạn** đã được điền
- Bài 3.3 (TODO) đã hoàn thiện